# Numerical Methods for Differential Equations

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/lorenzotomada/numerical_analysis_2026_2027/blob/main/labs/lab06/ODEs_SOLVED.ipynb)

We will now discuss strategies to approximate the solution to an ODE, which are ubiquitous in mathematical modeling.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

## Motivating example

We start with the following linear Cauchy model problem 
$$
\begin{aligned}
y^{\prime}(t) &= \lambda y(t)\\
y(0) & = 1
\end{aligned}
$$
whose exact solution is
$$
y(t) = e^{\lambda t}
$$

In [ ]:
l = -5.
t0 = 0.
tf = 10.
y0 = 1.

s = np.linspace(t0, tf, 5000)

exact = lambda x: np.exp(l*x)

### Forward Euler

$$
\frac{y_{n}-y_{n-1}}{h} = f(t_{n-1},y_{n-1})
$$

$$
y_n = y_{n-1} + h \lambda y_{n-1}
$$

In [ ]:
def forward_euler(l, y0, t0, tf, h):
        timesteps = np.arange(t0, tf+h, h)
        sol = np.zeros_like(timesteps)
        sol[0] = y0
        for i in range(1, len(sol)):
                sol[i] = sol[i-1] * (1 + l * h)

        return sol, timesteps
def rel_error(approx, exact):
    return np.linalg.norm(exact - approx, 2)/np.linalg.norm(exact, 2)

In [ ]:
fig, axs = plt.subplots(1, 3, figsize=(12, 3))

for (ax, dt) in zip(axs, [0.4, 0.25, 0.1]):
	
	# Solve the ODE using the forward Euler method
	y, t = forward_euler(l, y0, t0, tf, dt)

	# Plot the solution
	ax.plot(t, y, 'o-', label="Forward Euler")
	ax.plot(s, exact(s), label="Exact")
	ax.grid()
    
	# Add labels
	ax.set_xlabel('t')
	ax.set_ylabel('y')
	ax.legend()

	# Add a title
	ax.set_title(f'Forward Euler with dt = {dt}')

	# Compute the error
	error = rel_error(y, exact(t))
	print(error)

plt.show()

Indeed, we know the stability condition for Forward Euler for this problem is 
$$
h < \frac{2}{\vert \lambda \vert} = 0.4.
$$

### Backward Euler

$$
\frac{y_{n}-y_{n-1}}{h} = f(t_n,y_{n})
$$

$$
y_n = y_{n-1} + h \lambda y_{n}
$$
$$
y_{n} = \frac{y_{n-1}}{1 - h\lambda }
$$



In [ ]:
def backward_euler(l, y0, t0, tf, h):
    """Implement Backward Euler"""
    timesteps = np.arange(t0, tf + h, h)
    sol = np.zeros_like(timesteps)
    sol[0] = y0
    for i in range(1, len(sol)):
        sol[i] = sol[i-1] / (1 - l * h)

    return sol, timesteps

In [ ]:
fig, axs = plt.subplots(1, 3, figsize=(12, 3))

for (ax,dt) in zip(axs,[0.4, 0.25, 0.1]):
	
    # Solve the ODE using the backward Euler method
	y, t = backward_euler(l, y0, t0, tf, dt)

	# Plot the solution
	ax.plot(t, y, 'o-', label="Backward Euler")
	ax.plot(s, exact(s), label="Exact")
	ax.grid()

	# Add labels
	ax.set_xlabel('t')
	ax.set_ylabel('y')
	ax.legend()

	# Add a title
	ax.set_title(f'Backward Euler with dt = {dt}')

	# Compute the error
	error = rel_error(y, exact(t))
	print(error)

plt.show()

Compute the relative error varying the time step size.

In [ ]:
errors_forward = []
errors_backward = []
dts = np.logspace(np.log10(0.001), np.log10(0.5), num=100)
for dt in dts:
    y_forward, t = forward_euler(l, y0, t0, tf, dt)
    y_backward, t = backward_euler(l, y0, t0, tf, dt)
    errors_forward.append(rel_error(y_forward, exact(t)))
    errors_backward.append(rel_error(y_backward, exact(t)))

plt.figure(figsize=(4, 3))
plt.loglog(dts, errors_forward, label="Forward Euler")
plt.loglog(dts, errors_backward, label="Backward Euler")
plt.legend()
plt.grid()

## A more general problem

The previous implementation of the algorithms was specific to the given problem. Let us now implement the algorithms for generic forcing terms.

Using the new implementation we will solve the problem 

$$
\begin{cases}
y'(t) = 1 - y^2,\quad t>1 \\
y(1) = \frac{e^2 - 1}{e^2 + 1}
\end{cases},
$$

the exact solution of which is $y(t) = \frac{e^{2t} - 1}{e^{2t} + 1}$.


#### Forward Euler Method

The Forward Euler method for a generic forcing term $f(t,y)$ is given by:
$$
y_{n+1} = y_n + h f(t_n,y_n)
$$


In [ ]:
def forward_euler(f, y0, t0, tf, h):
	"""
	Implement Forward Euler
	f: f(t, y) function that defines the ODE
	y0: initial condition (array)
	t0: initial time
	tf: final time
	h: time step
	"""
	timesteps = np.arange(t0, tf + h, h)
	sol = np.zeros_like(timesteps)
	sol[0] = y0
	
	for i in range(1, len(sol)):
		sol[i] = sol[i-1] + h * f(timesteps[i-1], sol[i-1])
	
	return sol, timesteps

In [ ]:
# Define the function f(t, y) for the given problem
f = lambda t,y : 1 - y**2

# Problem parameters
y0 = (np.exp(2) - 1)/(np.exp(2) + 1)
t0 = 1
tf = 20
dt = 0.1
tt = np.linspace(t0, tf, 1000)

exact = lambda x: (np.exp(2*x) - 1)/(np.exp(2*x) + 1)

# Solve the ODE using the forward Euler method
y, t = forward_euler(f, y0, t0, tf, dt)

# Plot the solution
plt.figure(figsize=(8, 2))
plt.plot(t, y, '.-', label='Forward')
plt.plot(tt, exact(tt), label='Exact')

plt.xlabel('t')
plt.ylabel('y')
plt.grid()
plt.legend()
plt.show()


#### Backward Euler Method

The Backward Euler method for a generic forcing term $f(t,y)$ is given by:
$$
y_{n+1} = y_n + h f(t_{n+1}, y_{n+1})
$$

If $f$ is nonlinear in $y$, we cannot solve explicitly for $y_{n+1}$. Bringing everything to one side, at each step we must find the zero of
$$
g(y_{n+1}) := y_{n+1} - y_n - h f(t_{n+1}, y_{n+1}) = 0,
$$
i.e. a nonlinear zero-finding problem (e.g. Newton's method, or `scipy.optimize.fsolve`), using $y_n$ as initial guess.

If instead the ODE is linear, i.e. $f(t,y) = A y + b(t)$, this boils down to solving at each step the linear system
$$
(I - hA)\, y_{n+1} = y_n + h\, b(t_{n+1}).
$$

In [ ]:
from scipy.optimize import fsolve

def backward_euler(f, y0, t0, tf, h):
	"""
	Implement Backward Euler
	f: f(t,y) function that defines the ODE
	y0: initial condition
	t0: initial time
	tf: final time
	h: time step
	"""
	
	timesteps = np.arange(t0, tf + h, h)
	sol = np.zeros_like(timesteps)
	sol[0] = y0

	for i in range(1, len(sol)):
		# Define the implicit function for fsolve
		g = lambda y_next: y_next - sol[i-1] - h * f(timesteps[i], y_next)
		# Use fsolve to find the root of g, starting from the previous value
		sol[i] = fsolve(g, sol[i-1])[0]

	return sol, timesteps

In [ ]:
dt = 0.1
# Solve the ODE using the backward Euler method
y_be, t_be = backward_euler(f, y0, t0, tf, dt)

# Plot the solution
plt.figure(figsize=(8, 2))
plt.plot(t_be, y_be, '.-', label='Backward')
plt.plot(tt, exact(tt), label='Exact')

plt.xlabel('t')
plt.ylabel('y')
plt.grid()
plt.legend()
plt.show()

Plot the convergence of both methods varying the time step size.

In [ ]:
errors_forward = []
errors_backward = []
dts = np.logspace(np.log10(0.1), np.log10(1.), num=100)
for dt in dts:
    y_forward, t = forward_euler(f, y0, t0, tf, dt)
    y_backward, t = backward_euler(f, y0, t0, tf, dt)
    errors_forward.append(rel_error(y_forward, exact(t)))
    errors_backward.append(rel_error(y_backward, exact(t)))

plt.figure(figsize=(4, 3))
plt.loglog(dts, errors_forward, label="Forward Euler")
plt.loglog(dts, errors_backward, label="Backward Euler")
plt.legend()
plt.grid()

### Generalized model problem

Consider now the generalized model problem
$$
\begin{cases}
y'(t) = \lambda(t)\, y(t) + r(t), & t \in (0, +\infty),\\
y(0) = 1,
\end{cases}
\qquad \text{with } -\lambda_{\max} \le \lambda(t) \le -\lambda_{\min}, \quad 0 < \lambda_{\min} \le \lambda_{\max} < +\infty.
$$
Now the exact solution does not necessarily tend to zero (e.g. for constant $\lambda$ and $r$, $y(t) \to -r/\lambda$), so absolute stability is not the right requirement. Instead, we ask that **perturbations stay bounded**: if $z_n$ is the Forward Euler solution perturbed by $\rho_0$ on the initial datum and by $\rho_{n+1}$ at each step, with $|\rho_n| \le \rho$, then
$$
h < \frac{2}{\lambda_{\max}} \quad \Longrightarrow \quad |z_n - u_n| \le C\,\rho \quad \forall n,
$$
with $C$ independent of $n$ and $h$. If $h > 2/\lambda_{\max}$, perturbations are amplified as $n$ grows.

The same holds for a nonlinear problem $y' = f(t, y)$ if
$$
-\lambda_{\max} < \frac{\partial f}{\partial y}(t, y) < -\lambda_{\min} \qquad \forall t \ge 0,\ \forall y \in D_y,
$$
where $D_y$ contains the trajectory of the solution.

This restriction does not apply to Backward Euler, which keeps perturbations bounded for every $h > 0$. The condition $h < 2/\lambda_{\max}$ is a real issue, for example, for **stiff problems**: there $\lambda_{\max}$ is very large, so Forward Euler needs a tiny step size even when the solution is smooth and slowly varying. The step is then dictated by stability, not by accuracy.

Two examples, where we compare Forward and Backward Euler for two step sizes:

- $y' = \arctan(3y) - 3y + t$, $y(0) = 1$: $y(t) > 0$ and $\partial f/\partial y = \frac{3}{1+9y^2} - 3 \in (-3, 0)$, so $\lambda_{\max} = 3$ and Forward Euler needs $h < 2/3$. We perturb only the initial datum ($\rho_0 = 3$) and look at $e_n = z_n - u_n$.
- $y' = 1 - y^2$ (the problem considered above): $\partial f/\partial y = -2y \in (-2, -2y(1))$, so $\lambda_{\max} = 2$ and Forward Euler needs $h < 1$.

In [ ]:
# Example 1: arctan problem, perturbation of the initial datum
f_atan = lambda t, y: np.arctan(3*y) - 3*y + t
rho0 = 3.

fig, axs = plt.subplots(1, 2, figsize=(12, 3))
for ax, h in zip(axs, [2/3 - 0.01, 2/3 + 0.02]):
    for method, name in [(forward_euler, 'Forward Euler'), (backward_euler, 'Backward Euler')]:
        u, t = method(f_atan, 1., 0., 50., h)
        z, _ = method(f_atan, 1. + rho0, 0., 50., h)
        ax.plot(t, z - u, '.-', label=name)
    ax.set_xlabel('t')
    ax.set_ylabel('$e_n = z_n - u_n$')
    ax.set_title(f'h = 2/3 {"-" if h < 2/3 else "+"} {abs(h - 2/3):.2f}')
    ax.grid()
    ax.legend()
plt.show()

# Example 2: y' = 1 - y^2 on (1, 20)
f_ex2 = lambda t, y: 1 - y**2
y0_ex2 = (np.exp(2) - 1)/(np.exp(2) + 1)
tt_ex2 = np.linspace(1, 20, 1000)

fig, axs = plt.subplots(1, 2, figsize=(12, 3))
for ax, h in zip(axs, [20/21, 20/19]):
    for method, name in [(forward_euler, 'Forward Euler'), (backward_euler, 'Backward Euler')]:
        u, t = method(f_ex2, y0_ex2, 1., 20., h)
        ax.plot(t, u, '.-', label=name)
    ax.plot(tt_ex2, (np.exp(2*tt_ex2) - 1)/(np.exp(2*tt_ex2) + 1), 'k--', label='Exact')
    ax.set_xlabel('t')
    ax.set_ylabel('y')
    ax.set_title(f'h = {h:.3f}')
    ax.grid()
    ax.legend()
plt.show()

With implicit methods the time step can be chosen based on accuracy only. The price to pay is the solution of a (possibly nonlinear) equation at each step.

## A system of nonlinear ODEs

Let us consider the Lotka-Volterra predator-prey model for two interacting populations $x$ and $y$:
$$
\begin{cases}
	\dot{x} = \alpha x - \beta xy\\
	\dot{y} = -\gamma y + \delta xy \\
	x(t_0)=x_0,\ y(t_0)=y_0.
\end{cases}
$$
Here $x$ represent the population density of a _prey_ species, and $y$ represents the population density of a _predator_ species. 
The parameters $\alpha$, $\beta$, $\gamma$, and $\delta$ are positive constants that describe the interaction between the two species. Specifically:
- $\alpha$ is the growth rate of the prey population in the absence of predators.
- $\beta$ is the rate at which predators reduce the prey population.
- $\gamma$ is the death rate of the predator population in the absence of prey.
- $\delta$ is the rate at which the predator population increases by consuming prey.


In [ ]:
def F_Lotka_Volterra(t, Y):
	"""
	Define the Lotka-Volterra system of ODEs
	t: time
	Y: array with the values of the two variables
	"""

	x, y = Y
	dx = alpha * x - beta * x * y
	dy = delta * x * y - gamma * y

	return np.array([dx, dy])

In [ ]:
def fe_system(f, y0, t0, tf, h):
	"""
	Implement Forward Euler for a 2D system
	f: f(t,y) function that defines the ODE
	y0: initial condition
	t0: initial time
	tf: final time
	h: time step
	"""

	timesteps = np.arange(t0, tf + h, h)
	sol = np.zeros((len(timesteps), len(y0)))
	sol[0] = y0

	for i in range(1, len(sol)):
		sol[i] = sol[i-1] + h * f(timesteps[i-1], sol[i-1])

	return sol, timesteps


In [ ]:
def be_system(f, y0, t0, tf, h):
	"""
	Implement Backward Euler for a 2D system
	f: f(t,y) function that defines the ODE
	y0: initial condition
	t0: initial time
	tf: final time
	h: time step
	"""

	timesteps = np.arange(t0, tf + h, h)
	sol = np.zeros((len(timesteps), len(y0)))
	sol[0] = y0

	for i in range(1, len(sol)):
		# Define the implicit function for fsolve
		g = lambda y_next: y_next - sol[i-1] - h * f(timesteps[i], y_next)
		# Use fsolve to find the root of g, starting from the previous value
		sol[i] = fsolve(g, sol[i-1])

	return sol, timesteps

In [ ]:

# Initial conditions
y0 = np.array([40, 20])
t0 = 0
tf = 100
dt = 0.5

# Parameters
alpha = 0.08
beta = 0.004
gamma = 0.06	
delta = 0.002

# Solve the ODE using the forward Euler method
y_fe, t_fe = fe_system(F_Lotka_Volterra, y0, t0, tf, dt)

# Solve the ODE using the backward Euler method
y_be, t_be = be_system(F_Lotka_Volterra, y0, t0, tf, dt)

# Plot the solution
fig, axs = plt.subplots(1, 2, figsize=(12, 6))

# Plot components
axs[0].plot(t_fe, y_fe[:,0], label='Prey (Forward Euler)', color='blue')
axs[0].plot(t_fe, y_fe[:,1], label='Predator (Forward Euler)', color='blue', linestyle='--')
axs[0].plot(t_be, y_be[:,0], label='Prey (Backward Euler)', color='red')
axs[0].plot(t_be, y_be[:,1], label='Predator (Backward Euler)', color='red', linestyle='--')
axs[0].set_xlabel('Time')
axs[0].legend()
axs[0].grid()

# Plot phase space
X, Y = np.meshgrid(np.linspace(min(y_fe[:,0]), max(y_fe[:,0]), 20), np.linspace(min(y_fe[:,1]), max(y_fe[:,1]), 20))
U = alpha * X - beta * X * Y
V = delta * X * Y - gamma * Y
axs[1].quiver(X, Y, U, V, color='gray', alpha=0.5)
axs[1].plot(y_fe[:,0], y_fe[:,1], label='Forward Euler', color='blue')
axs[1].plot(y_be[:,0], y_be[:,1], label='Backward Euler', color='red')
axs[1].plot(y0[0],y0[1],"ro")
axs[1].set_xlabel('Prey Population')
axs[1].set_ylabel('Predator Population')
axs[1].legend(loc='upper left')
axs[1].grid()
plt.show()

## Crank-Nicolson Method

$$
u_{n+1} = u_n + \frac{h}{2} \left(f(t_n,u_n) + f(t_{n+1},u_{n+1})\right)
$$

In [ ]:
def cn_system(f, y0, t0, tf, h):
	"""
	Implement Crank-Nicolson for a system
	f: f(t,y) function that defines the ODE
	y0: initial condition
	t0: initial time
	tf: final time
	h: time step
	"""

	timesteps = np.arange(t0, tf + h, h)
	sol = np.zeros((len(timesteps), len(y0)))
	sol[0] = y0

	for i in range(1, len(sol)):
		# Define the implicit function for fsolve
		g = lambda y_next: y_next - sol[i-1] - 0.5 * h * (f(timesteps[i-1], sol[i-1]) + f(timesteps[i], y_next))
		# Use fsolve to find the root of g, starting from the previous value
		sol[i] = fsolve(g, sol[i-1])

	return sol, timesteps

In [ ]:

# Initial conditions
y0 = np.array([40, 20])
t0 = 0
tf = 500
dt = 0.5

# Parameters
alpha = 0.08
beta = 0.004
gamma = 0.06	
delta = 0.002

# Solve the ODE using the forward Euler method
y_fe, t_fe = fe_system(F_Lotka_Volterra, y0, t0, tf, dt)

# Solve the ODE using the backward Euler method
y_be, t_be = be_system(F_Lotka_Volterra, y0, t0, tf, dt)

# Solve the ODE using the Crank-Nicolson method
y_cn, t_cn = cn_system(F_Lotka_Volterra, y0, t0, tf, dt)


# Plot the solution
fig, axs = plt.subplots(1, 2, figsize=(12, 6))

# Plot components
axs[0].plot(t_fe, y_fe[:,0], label='Prey (Forward Euler)', color='blue')
axs[0].plot(t_fe, y_fe[:,1], label='Predator (Forward Euler)', color='blue', linestyle='--')
axs[0].plot(t_be, y_be[:,0], label='Prey (Backward Euler)', color='red')
axs[0].plot(t_be, y_be[:,1], label='Predator (Backward Euler)', color='red', linestyle='--')
axs[0].plot(t_cn, y_cn[:,0], label='Prey (Crank-Nicolson)', color='green')
axs[0].plot(t_cn, y_cn[:,1], label='Predator (Crank-Nicolson)', color='green', linestyle='--')
axs[0].set_xlabel('Time')
axs[0].legend()
axs[0].grid()

# Plot phase space
X, Y = np.meshgrid(np.linspace(min(y_fe[:,0]), max(y_fe[:,0]), 20), np.linspace(min(y_fe[:,1]), max(y_fe[:,1]), 20))
U = alpha * X - beta * X * Y
V = delta * X * Y - gamma * Y
axs[1].quiver(X, Y, U, V, color='gray', alpha=0.5)
axs[1].plot(y_fe[:,0], y_fe[:,1], label='Forward Euler', color='blue')
axs[1].plot(y_be[:,0], y_be[:,1], label='Backward Euler', color='red')
axs[1].plot(y_cn[:,0], y_cn[:,1], label='Crank-Nicolson', color='green')
axs[1].plot(y0[0],y0[1],"ro")
axs[1].set_xlabel('Prey Population')
axs[1].set_ylabel('Predator Population')
axs[1].legend(loc='upper left')
axs[1].grid()
plt.show()